# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
([FEA-026: deterministic object identity](../../../mbse-basis/docs/features/FEA-026.md)). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. An object is a singleton, owned, or counted: a counted
object has no identity beyond the document that writes it, which numbers it.

In [ ]:
import { Builders as B, JSON as Json, Paths, Plain, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, KeyError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, listed, map, raises, same, view } from "./support.js";

/** The directory of definitions, each owning its clauses: a singleton wts.Features owns each wts.Definition through
 * wts.Defines, by a name its pattern gives, and a definition each wts.Clause through wts.HasClause. */
function wts(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("wts/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Directory: S.OfObject, Definition: S.OfObject, Clause: S.OfObject, Defines: S.OfRelation,
      HasClause: S.OfRelation })
    .spec([
      (_: any) => _.Directory.name("wts.Directory").ref(true).singleton("wts.Features").adjacencies([
        (t: any) => t.name("dir").relation([_.Defines]).me("directory")]),
      (_: any) => _.Definition.name("wts.Definition").ref(true).properties([
        (t: any) => t.name("title").schema([S.OfStr]),
      ]).adjacencies([
        (t: any) => t.name("def").relation([_.Defines]).me("defn").owner(true),
        (t: any) => t.name("clauses").relation([_.HasClause]).me("defn"),
      ]),
      (_: any) => _.Clause.name("wts.Clause").ref(true).adjacencies([
        (t: any) => t.relation([_.HasClause]).me("clause").owner(true)]),
      (_: any) => (
        _.Defines
          .name("wts.Defines")
          .links([
            (t: any) => t.name("directory"),
            (t: any) => t.name("defn"),
          ])
          .properties([(t: any) => t.name("name").schema([S.OfStr]).pattern(String.raw`DEF-(\d+\d\d)`)])
          .uniques([["directory", "name"], ["defn"]])
      ),
      (_: any) => (
        _.HasClause
          .name("wts.HasClause")
          .links([
            (t: any) => t.name("defn"),
            (t: any) => t.name("clause"),
          ])
          .properties([(t: any) => t.name("name").schema([S.OfStr])])
          .uniques([["defn", "name"], ["clause"]])
      ),
    ])]))]);
}

const store: any = new B.Store();
const Text = store.Schemas.OfStr.as_schema(); // the store's own
store.transaction(wts);
const [Directory, Definition, Clause] = [store.registered("wts.Directory"), store.registered("wts.Definition"), store.registered("wts.Clause")];
const [Defines, HasClause] = [store.registered("wts.Defines"), store.registered("wts.HasClause")];

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
{
  store.transaction(() => {
    assert(Definition.adjacencies.get("def").owner === true && Directory.adjacencies.get("dir").owner === null);
    assert([Directory, Definition, Clause, Defines, HasClause].every((s) => s.validate().length === 0));
    // An object schema owned through a relation, each validated: the relation's links and unique clauses decide.
    assert(store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([(t: any) => t.name("up").relation([
      store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("up"), (t: any) => t.name("down")]).uniques([["up"], ["down"]])),
    ]).me("down").owner(true)])).validate().length === 0); // keyless: the owner alone determines the owned object
    assert(same(store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([(t: any) => t.name("up").relation([
      store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("up"), (t: any) => t.name("down")]).uniques([["down"]])),
    ]).me("down").owner(true)])).validate(), [
      "adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that the owned object "
      + "determines its owner"]));
    assert(same(store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([(t: any) => t.name("up").relation([
      store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("up"), (t: any) => t.name("down")]).uniques([["up"]])),
    ]).me("down").owner(true)])).validate(), [
      "adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that the owner and the keys "
      + "determine the owned object"]));
    assert(same(store.Schemas.OfObject((o: any) => o.ref(true).adjacencies([(t: any) => t.name("up").relation([
      store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("up"), (t: any) => t.name("down"), (t: any) => t.name("side")])
        .uniques([["up"], ["down"]])),
    ]).me("down").owner(true)])).validate(), [
      "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]));
    raises(TypeError, () => store.Schemas.OfObject((o: any) => o.adjacencies([(t: any) => t.relation([Defines]).me("defn").owner("yes")])),
      "property 'owner' holds bool, got str");
  });
}

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
{
  store.transaction(() => {
    const name = Defines.entry.properties.get("name");
    assert(name.pattern === String.raw`DEF-(\d+\d\d)` && name.name_for(7) === "DEF-007" && name.name_for(1234) === "DEF-1234");
    assert(same([name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")], [693, null, null]));
    assert(new S.OfProperty.Data({ name: "k", schema: Text }).number_of("DEF-693") === null); // no pattern
    const bad = store.Schemas.OfRelation((r: any) => (
      r
        .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
        .properties([
          (t: any) => t.name("k").schema([Text]).pattern("DEF-.*"),
          (t: any) => t.name("n").schema([store.Schemas.OfInt]).pattern(String.raw`(\d+)`),
        ])
    ));
    assert(same(bad.validate(), [
      "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
      "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]));
  });
}

## IDN-03 · Every store holds the path schemas; the members of its own relations are owned

In [ ]:
{
  for (const name of ["Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"]) {
    assert(store.registered(name).validate().length === 0 && store.framework(store.registered(name)));
  }
  assert(same([...store.Paths.OfRoot.as_schema().properties.keys()], ["singleton", "next"]));
  assert(same([...store.Paths.OfEntry.as_schema().properties.keys()], ["i", "o", "properties", "next"]));
  assert(same(store.Paths.Next.as_schema().branches.map((b: any) => b.name), ["entry", "target"]));
  const other: any = new B.Store();
  assert(other.Paths.OfRoot.as_schema() !== store.Paths.OfRoot.as_schema()); // each store's own
  assert(!store.Schemas.OfObject.as_schema().adjacencies.has("instances")); // a schema holds no instances
  const owning = store.Schemas.OfProperty.as_schema().anonymous.filter((a: any) => a.owner).map((a: any) => `${a.relation.name} ${a.me}`);
  assert(same(owning, ["Schemas.HasProperty property"]));
  assert(store.Schemas.OfObject.as_schema().anonymous.filter((a: any) => a.relation.name === "Schemas.Defines").every((a: any) => a.owner));
}

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
const features = store.Instances.wts.Features;
const toPlain = (value: unknown) => Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), value);
{
  assert(features === store.singleton("wts.Features"));
  store.transaction(() => {
    store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.name("DEF-693").defn((d: any) => (
      d
        .title("Has 88 keys")
        .clauses((c: any) => c.insert((h: any) => h.name("C-1").clause((x: any) => x)))
    )))));
  });
  const definition = features.dir({ name: "DEF-693" });
  const clause = definition.clauses({ name: "C-1" });
  assert(equal(toPlain(Paths.of(definition)), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { target: {} } } } })));
  assert(equal(toPlain(Paths.of(clause)), map({ singleton: "wts.Features", next: { entry: {
    i: "dir", o: "def", properties: { name: "DEF-693" }, next: { entry: {
      i: "clauses", o: "", properties: { name: "C-1" }, next: { target: {} } } } } } }))); // an anonymous owner
  assert(equal(toPlain(Paths.of(features)), map({ singleton: "wts.Features", next: { target: {} } })));
  assert(Paths.same(Paths.of(clause), Paths.of(clause)) && !Paths.same(Paths.of(clause), Paths.of(definition)));
  assert(Paths.same(null, null) && !Paths.same(Paths.of(clause), null));
  store.transaction(() => {
    store.wts.Definition(definition, (d: any) => d.clauses((a: any) => a.remove((e: any) => e.clause(clause))));
  });
  raises(LookupError, () => Paths.of(clause), "a 'wts.Clause' has no owner: its path is not known"); // its owning entry removed
  /** wts.Twice, owned through wts.Holds by another wts.Twice. */
  const twice = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("twice/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Twice: S.OfObject, Holds: S.OfRelation })
      .spec([
        (_: any) => _.Twice.name("wts.Twice").ref(true).adjacencies([
          (t: any) => t.name("up").relation([_.Holds]).me("b").owner(true),
          (t: any) => t.name("down").relation([_.Holds]).me("a"),
        ]),
        (_: any) => _.Holds.name("wts.Holds").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([["a"], ["b"]]),
      ])]))]);
  };
  store.transaction(twice);
  const [first] = store.transaction(() => { // within one transaction, objects are given their owners when it ends: here, each other
    const [first, second] = [store.wts.Twice((t: any) => t), store.wts.Twice((t: any) => t)];
    store.wts.Twice(first, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(second))));
    store.wts.Twice(second, (t: any) => t.up((a: any) => a.insert((e: any) => e.a(first))));
    return [first, second];
  });
  raises(LookupError, () => Paths.of(first), "an ownership cycle: no singleton owns the object");
  /** wts.Split, which wts.Features may own two ways, through wts.Lefts and wts.Rights. */
  const splits = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("split/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Lefts: S.OfRelation, Rights: S.OfRelation, Directory: S.OfObject, Split: S.OfObject })
      .spec([ // named: each end's adjacency links it
        (_: any) => _.Lefts.name("wts.Lefts").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([["a"], ["b"]]),
        (_: any) => _.Rights.name("wts.Rights").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([["a"], ["b"]]),
        (_: any) => _.Directory.name("wts.Directory").adjacencies([(t: any) => t.relation([_.Lefts]).me("a"),
          (t: any) => t.name("rights").relation([_.Rights]).me("a")]),
        (_: any) => _.Split.name("wts.Split").ref(true).adjacencies([
          (t: any) => t.name("left").relation([_.Lefts]).me("b").owner(true),
          (t: any) => t.name("right").relation([_.Rights]).me("b").owner(true),
        ]),
      ])]))]);
  };
  store.transaction(splits);
  const [split] = store.transaction(() => {
    const split = store.wts.Split((s: any) => s.left((a: any) => a.insert((e: any) => e.a(features))));
    return [split];
  });
  assert(equal(toPlain(Paths.of(split)), map({ singleton: "wts.Features", next: { entry: {
    i: "", o: "left", properties: {}, next: { target: {} } } } }))); // the owner's adjacency anonymous
  store.transaction(() => {
    store.wts.Split(split, (s: any) => s.right((a: any) => a.insert((e: any) => e.a(features))));
  });
  raises(LookupError, () => Paths.of(split), "a 'wts.Split' has more than one owner: its path is not known");
  store.transaction(() => {
    raises(ReadOnlyError, () => store.wts.Twice(first, (t: any) => t.as_identity("twice-9")), // a program writes no identity
      "'as_identity' is not a property or adjacency of 'wts.Twice'");
  });
}

## IDN-05 · An object built without an owner is given the one its schema allows: a singleton's, its keys from their naming patterns; else refused. Removing its owning entry deletes it, and what it owns

In [ ]:
{
  const [made] = store.transaction(() => {
    store.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.defn((d: any) => d.title("nameless"))))); // no key
    const made = store.wts.Definition((d: any) => d.title("made alone")); // its owner the one singleton that can be, Features
    return [made];
  });
  assert(features.dir({ name: "DEF-694" }) === made); // its key the next its pattern gives, past every number held
  const defn = store.transaction(() => store.wts.Definition((d: any) => d));
  assert(defn === features.dir({ name: "DEF-695" }));
  raises(ValueError, () => store.transaction(() => store.wts.Clause((c: any) => c)), // a definition owns it, and none is a singleton
    "a 'wts.Clause' needs an owner through 'wts.HasClause': give it one, since no singleton can be");
  raises(ValueError, () => store.transaction(() => store.wts.Split((s: any) => s)),
    "a 'wts.Split' needs an owner through 'left', 'right': give it one, since more than one singleton can be");
  /** wts.Note, owned by wts.Features through wts.Notes, by a name. */
  const notes = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("notes/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Notes: S.OfRelation, Directory: S.OfObject, Note: S.OfObject })
      .spec([
        (_: any) => (
          _.Notes
            .name("wts.Notes")
            .links([(t: any) => t.name("directory"), (t: any) => t.name("note")])
            .properties([(t: any) => t.name("name").schema([S.OfStr])]) // a key with no naming pattern
            .uniques([["directory", "name"], ["note"]])
        ),
        (_: any) => _.Directory.name("wts.Directory").adjacencies([(t: any) => t.name("notes").relation([_.Notes]).me("directory")]),
        (_: any) => _.Note.name("wts.Note").ref(true).adjacencies([
          (t: any) => t.relation([_.Notes]).me("note").owner(true),
        ]),
      ])]))]);
  };
  store.transaction(notes);
  raises(ValueError, () => store.transaction(() => store.wts.Note((n: any) => n)),
    "a 'wts.Note' needs an owner: the key 'name' of 'wts.Notes' has no naming pattern to give it a value");
  store.transaction(() => {
    store.wts.Directory(features, (f: any) => f.notes((a: any) => a.insert((e: any) => e.name("first").note((n: any) => n))));
  });
  assert(features.notes({ name: "first" }).as_schema() === store.registered("wts.Note"));

  // Removing an owning entry deletes the owned object: every entry it is in goes, and so, in turn, what it owned.
  const [gone] = store.transaction(() => {
    const gone = store.wts.Definition((d: any) => d.title("gone").clauses((c: any) => c.insert((h: any) => h.name("C-1").clause((x: any) => x))));
    return [gone];
  });
  const its = gone.clauses({ name: "C-1" });
  store.transaction(() => {
    store.wts.Directory(features, (f: any) => f.dir((a: any) => a.remove((e: any) => e.defn(gone))));
  });
  assert(gone.clauses.length === 0 && [...features.dir].every((e: any) => e.defn !== gone));
  raises(LookupError, () => Paths.of(its), "a 'wts.Clause' has no owner: its path is not known"); // deleted with the definition that owned it

  // Nothing is owned by a counted object: an object neither a singleton nor owned.
  /** wts.Loose, counted, which would own a wts.Kept through wts.Keeps. */
  const kept = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("kept/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Loose: S.OfObject, Keeps: S.OfRelation, Kept: S.OfObject })
      .spec([
        (_: any) => _.Loose.name("wts.Loose").ref(true).adjacencies([(t: any) => t.name("keeps").relation([_.Keeps]).me("keeper")]),
        (_: any) => _.Keeps.name("wts.Keeps").links([(t: any) => t.name("keeper"), (t: any) => t.name("kept")]).uniques([
          ["keeper"], ["kept"]]),
        (_: any) => _.Kept.name("wts.Kept").ref(true).adjacencies([(t: any) => t.relation([_.Keeps]).me("kept").owner(true)]),
      ])]))]);
  };
  store.transaction(kept);
  const Loose = store.registered("wts.Loose");
  raises(ValueError, () => store.transaction(() => store.wts.Loose((l: any) => l.keeps((a: any) => a.insert((e: any) => e.kept((k: any) => k))))),
    "a 'wts.Kept' cannot be owned by a counted 'wts.Loose': nothing is owned by a counted object");
  raises(ValueError, () => store.transaction(() => store.wts.Loose((l: any) => l.keeps((a: any) => a.insert((e: any) => e.kept((k: any) => k))))),
    "nothing is owned by a counted object"); // refused within another
  const [loose] = store.transaction(() => {
    const loose = store.wts.Loose((l: any) => l);
    return [loose];
  });
  assert(loose.as_schema() === Loose); // a refused write leaves nothing pending for the next
  assert(same(Validators.Validate(store)(Loose, loose), [ // and validation refuses the schemas that would allow it
    "schema 'wts.Loose': adjacency 'keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is owned "
    + "by a counted object, which a 'wts.Loose' is"]));
  /** wts.Stray, counted, which would own a wts.Kept through an anonymous adjacency. */
  const stray = (txn: any): any => txn.uid("stray/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Stray: txn.Schemas.OfObject })
    .spec([(_: any) => _.Stray.name("wts.Stray").ref(true).adjacencies([
      (t: any) => t.relation([txn.wts.Keeps]).me("keeper")])])]))]);
  store.transaction(stray);
  const Stray = store.registered("wts.Stray");
  store.transaction(() => {
    assert(same(Validators.Validate(store)(Stray, store.wts.Stray((s: any) => s)), [ // an anonymous adjacency, by its relation
      "schema 'wts.Stray': adjacency 'wts.Keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is "
      + "owned by a counted object, which a 'wts.Stray' is"]));
  });
  assert(Validators.Validate(store)(Directory, features).length === 0); // a singleton owns
}

## IDN-06 · Lookup by keys: on any adjacency; `store.Instances` finds only singletons, and a counted object has no path

In [ ]:
{
  /** wts.Free, with no owner() adjacency, and not a singleton: counted. */
  const counted = (txn: any): any => txn.uid("free/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Free: txn.Schemas.OfObject })
    .spec([(_: any) => _.Free.name("wts.Free").ref(true)])]))]);
  store.transaction(counted);
  const free = store.transaction(() => store.wts.Free((f: any) => f)); // no owner() adjacency, and not a singleton: counted
  raises(LookupError, () => Paths.of(free), "a 'wts.Free' is counted: it has no path");
  assert(repr(store.Instances.wts) === "<Instances.wts>");
  assert(store.Instances.wts.then === undefined && store.Instances.wts[Symbol.iterator] === undefined); // probes name no part
  raises(LookupError, () => store.Instances.wts.Free({ name: "free-1" }),
    "no singleton named 'wts.Free': an owned object is found through its owners' adjacencies, a counted one only through what links it");
  raises(LookupError, () => store.Instances.wts.Definition({ name: "DEF-693" }), // owned through Defines: found as features.dir({ name: ... })
    "no singleton named 'wts.Definition'");
  raises(AttributeError, () => store.Instances._hidden, "_hidden");
  raises(LookupError, () => features.dir({ name: "DEF-999" }), "no entry of 'dir' has the keys {'name': 'DEF-999'}");
  raises(KeyError, () => features.dir({ nope: 1n }), "'nope' is not a property of 'wts.Defines'");
  /** wts.Trio, a relation of three links, through which wts.Features has `trio`. */
  const trio = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("trio/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Trio: S.OfRelation, Directory: S.OfObject })
      .spec([
        (_: any) => _.Trio.name("wts.Trio").links([(t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]),
        (_: any) => _.Directory.name("wts.Directory").adjacencies([(t: any) => t.name("trio").relation([_.Trio]).me("a")]),
      ])]))]);
  };
  store.transaction(trio);
  raises(TypeError, () => features.trio({}), "a lookup through 'trio' needs a relation of two links");
}

## IDN-07 · Owned objects in documents and snapshots: a document refers to what it does not hold by its path; an owned root, placed by the entry that owns it; a snapshot carries them, each found again by the entry that owns it

In [ ]:
{
  const [source, target]: any[] = [new B.Store(), new B.Store()];
  source.transaction(wts);
  target.transaction(wts);
  const features = source.Instances.wts.Features;
  source.transaction(() => { // wts.Features, defining DEF-001, which holds the clause C-1
    source.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.name("DEF-001").defn((d: any) => (
      d
        .title("Keys")
        .clauses((c: any) => c.insert((h: any) => h.name("C-1").clause((x: any) => x)))
    )))));
  });
  const [Directory, Definition, Clause] = ["Directory", "Definition", "Clause"].map((name) => source.registered(`wts.${name}`));
  const graph: any = Plain.ToPlain(source).Reachable(Directory, features);
  let at: any = map({ singleton: "wts.Features", next: { entry: { i: "dir", o: "def", properties: { name: "DEF-001" }, next: {
    target: {} } } } });
  assert(equal(view(graph).get("objects"), [listed("wts.Directory", 0, { dir: [{ defn: { $ref: at }, name: "DEF-001" }] })])); // alone
  raises(LookupError, () => Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph),
    "no object of the store has the path a spec refers to"); // what it owns is a snapshot's to carry
  assert(Plain.FromPlain(source).Reachable(Directory, graph) === features && features.dir.length === 1); // held: found
  const defn = features.dir({ name: "DEF-001" });
  const written: any = Plain.ToPlain(source).OfObject(Definition, defn); // an owned root: the entry that owns it, its owner by path
  assert(equal(view(written).get("objects")[0].get("properties").get("def"), map([{ directory: { $ref: { singleton: "wts.Features",
    next: { target: {} } } }, name: "DEF-001" }])));
  assert(Plain.FromPlain(source).OfObject(Definition, written) === defn); // found by the entry that owns it
  raises(ValueError, () => Plain.ToPlain(source).OfObject(Clause, defn.clauses({ name: "C-1" })),
    "a 'wts.Clause' is owned through an adjacency with no name: a document cannot give the entry that owns it; name the adjacency");
  source.transaction(() => {
    source.wts.Directory(features, (f: any) => f.dir((a: any) => a.insert((e: any) => e.name("DEF-003").defn((d: any) => d.title("Bare")))));
  });
  const bare = Json.loads(Json.dumps(Plain.ToPlain(source).OfObject(Definition, features.dir({ name: "DEF-003" }))).replaceAll("DEF-003", "DEF-004"));
  const made: any = Plain.FromPlain(source).OfObject(Definition, bare); // an owned root none holds: made, placed by its owning entry
  assert(features.dir({ name: "DEF-004" }) === made && `${made.title}` === "Bare");
  const renamed = Json.loads(Json.dumps(graph).replaceAll('"o": "def"', '"o": "nope"'));
  raises(LookupError, () => Plain.FromPlain(source).Reachable(Directory, renamed),
    "no object of the store has the path a spec refers to"); // a path through no owning adjacency names nothing

  // A store's snapshot carries the owned objects: each a symbol, placed by the entry that owns it, its owner before it.
  const snapshot = Plain.ToPlain(source).Snapshot();
  const third: any = new B.Store();
  const read = Plain.FromPlain(third).Snapshot(snapshot);
  assert(Json.dumps(Plain.ToPlain(third).Snapshot() as never) === Json.dumps(snapshot as never));
  assert(third.Instances.wts.Features.dir({ name: "DEF-001" }).clauses({ name: "C-1" }).as_schema() === third.registered("wts.Clause"));
  const again = Plain.FromPlain(third).Snapshot(snapshot); // read again: each found, a clause through its definition's entry (60)
  assert(again.length === read.length && again.every((a, k) => a === read[k]));
  assert(same([...third.Instances.wts.Features.dir].map((e: any) => `${e.name}`), ["DEF-001", "DEF-003", "DEF-004"]));
}